# 🧠 Policy RAG Chatbot
### A Multi-Organization Policy Assistant
**Stack:** Python + LangChain + Google Gemini + Qdrant Cloud + Gradio

For the LLM/RAG fundamentals behind this project, see the companion notebook **`LLM_Fundamentals_Basics.ipynb`**.

**Scope (V1):** PDF policies only, one active version per (organization, policy), manual org/policy tagging (no auto-detection), **Qdrant Cloud** as the persistent vector store, no conversation history. Runs the same way in **Kaggle, Colab, or plain Jupyter** — uploads use `ipywidgets`, not a platform-specific API.

**Files needed alongside this notebook:** `policy_assistant.css` (Gradio UI styling — Section 2 reads it at runtime; upload it in the same folder on Kaggle/Colab).

**Before running:** create a free cluster at **[cloud.qdrant.io](https://cloud.qdrant.io)** (free tier: 1GB) and grab its cluster URL and an API key — you'll be prompted for both in Setup below.

## Setup

### Install packages

In [ ]:
# LangChain core + the Gemini integration + Google's own SDK (used for a couple of direct calls later)
# pypdf/langchain-community: PDF loading. ipywidgets: the file-upload widget in Section 1.6, works
# the same in Kaggle/Colab/Jupyter. gradio (pinned, since some hosted notebooks preinstall an older
# version that doesn't support type="messages" on Chatbot) is the demo UI. qdrant-client/langchain-qdrant:
# persistent, free-tier cloud vector database.
!pip install -q langchain langchain-core langchain-community langchain-google-genai google-generativeai langchain-text-splitters pypdf ipywidgets "gradio>=4.44.0" qdrant-client langchain-qdrant

### Provide your Gemini API key

We use `getpass` so the key is never echoed or saved into the notebook file — get one at **[Google AI Studio](https://aistudio.google.com/app/apikey)**.

In [ ]:
import os
from getpass import getpass

# You will be prompted to paste your key below. It will NOT be displayed or saved to the notebook.
if not os.environ.get("GOOGLE_API_KEY"):
    os.environ["GOOGLE_API_KEY"] = getpass("Enter your Gemini API key: ")

print("API key loaded into this session ✅ (not printed, not saved to the notebook file)")


API key loaded into this session ✅ (not printed, not saved to the notebook file)


### Provide your Qdrant Cloud credentials

From your cluster's page on [cloud.qdrant.io](https://cloud.qdrant.io): the **Cluster URL** (looks like `https://xxxxxxxx.us-east.aws.cloud.qdrant.io:6333`) and an **API key** (Cluster → API Keys → Create).

In [ ]:
if not os.environ.get("QDRANT_URL"):
    os.environ["QDRANT_URL"] = input("Enter your Qdrant Cloud cluster URL: ").strip()

if not os.environ.get("QDRANT_API_KEY"):
    os.environ["QDRANT_API_KEY"] = getpass("Enter your Qdrant Cloud API key: ")

print("Qdrant Cloud credentials loaded into this session ✅ (not printed, not saved to the notebook file)")

In [ ]:
from langchain_google_genai import ChatGoogleGenerativeAI

def get_text(response):
    """Return just the plain text of an LLM response, whether `.content` is a string
    or a list of content blocks (e.g. text + an internal thought-signature block)."""
    content = response.content
    if isinstance(content, str):
        return content
    return "".join(
        block.get("text", "")
        for block in content
        if isinstance(block, dict) and block.get("type") == "text"
    )

print("get_text() ready.")

## 1. Core Pipeline

```
PDF Upload → Page-aware Text Extraction → Organization/Policy Tagging → Chunking
   → Embeddings → Qdrant Cloud (persistent) → Organization-filtered Retrieval
   → Strict LLM Prompt → Answer + Citations (or the required "no answer" message)
```

Every chunk carries metadata (`organization`, `policy_name`, `policy_version`, `source_file`, `page`). Retrieval is filtered by organization, so one company's policies can never answer another's questions.

### 1.1 PDF loading (page-aware)

In [ ]:
from langchain_community.document_loaders import PyPDFLoader

def load_pdf_as_documents(pdf_path, organization, policy_name, policy_version):
    """Load a PDF page-by-page and tag every page with organization/policy metadata."""
    loader = PyPDFLoader(pdf_path)
    pages = loader.load()  # one Document per page, metadata already has "page"

    for page_doc in pages:
        page_doc.metadata.update({
            "organization": organization,
            "policy_name": policy_name,
            "policy_version": policy_version,
            "source_file": os.path.basename(pdf_path),
        })
    return pages

print("load_pdf_as_documents() ready — we'll use it once we have a PDF to try in the Gradio app below.")

### 1.2 Chunking (metadata-preserving)

In [ ]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

policy_splitter = RecursiveCharacterTextSplitter(chunk_size=800, chunk_overlap=100)

def chunk_documents(page_documents):
    return policy_splitter.split_documents(page_documents)

print("chunk_documents() ready.")

### 1.3 Embeddings + a shared Qdrant Cloud collection

One shared collection for every organization; isolation happens via metadata filtering at query time (Section 1.4), same as before — Qdrant just makes it persistent across sessions instead of living only in this runtime's RAM. `index_policy_pdf(...)` loads → tags → chunks → embeds → stores, and replaces any existing chunks for that (organization, policy_name) pair — different policies for the same org coexist; re-uploading the same policy replaces the old version. On startup we also scan the collection so `indexed_policies` reflects anything already uploaded in a previous session.

In [ ]:
from langchain_google_genai import GoogleGenerativeAIEmbeddings
from langchain_qdrant import QdrantVectorStore
from qdrant_client import QdrantClient
from qdrant_client.models import Distance, VectorParams, Filter, FieldCondition, MatchValue, FilterSelector, PayloadSchemaType
import google.generativeai as genai

genai.configure(api_key=os.environ["GOOGLE_API_KEY"])

DEFAULT_EMBEDDING_MODEL = "models/gemini-embedding-001"

def _resolve_embedding_model():
    # genai.list_models() occasionally times out over Colab's network — fall back to the
    # known default instead of failing the whole pipeline on a transient network hiccup.
    try:
        return next(m.name for m in genai.list_models() if "embedContent" in m.supported_generation_methods)
    except Exception as e:
        print(f"Could not list models ({e}); falling back to default '{DEFAULT_EMBEDDING_MODEL}'.")
        return DEFAULT_EMBEDDING_MODEL

embedding_model_name = _resolve_embedding_model()
print("Using embedding model:", embedding_model_name)

embeddings = GoogleGenerativeAIEmbeddings(model=embedding_model_name)

QDRANT_COLLECTION = "policy_documents"
qdrant_client = QdrantClient(url=os.environ["QDRANT_URL"], api_key=os.environ["QDRANT_API_KEY"])

if not qdrant_client.collection_exists(QDRANT_COLLECTION):
    vector_size = len(embeddings.embed_query("dimension probe"))
    qdrant_client.create_collection(
        collection_name=QDRANT_COLLECTION,
        vectors_config=VectorParams(size=vector_size, distance=Distance.COSINE),
    )
    print(f"Created Qdrant collection '{QDRANT_COLLECTION}'.")
else:
    print(f"Using existing Qdrant collection '{QDRANT_COLLECTION}'.")

# Qdrant requires an explicit payload index before you can filter/delete on a field.
# Safe to call every run — it's a no-op if the index already exists.
for _field in ("metadata.organization", "metadata.policy_name"):
    try:
        qdrant_client.create_payload_index(
            collection_name=QDRANT_COLLECTION,
            field_name=_field,
            field_schema=PayloadSchemaType.KEYWORD,
        )
    except Exception:
        pass

policy_vector_store = QdrantVectorStore(client=qdrant_client, collection_name=QDRANT_COLLECTION, embedding=embeddings)

# Track indexed policies per (organization, policy_name) — an org can have several distinct
# policies (Leave, WFH, Travel, ...); only re-indexing the SAME policy_name replaces it.
indexed_policies = {}  # (organization, policy_name) -> {"policy_version", "source_file", "chunk_count"}

def _org_policy_filter(organization, policy_name=None):
    conditions = [FieldCondition(key="metadata.organization", match=MatchValue(value=organization))]
    if policy_name is not None:
        conditions.append(FieldCondition(key="metadata.policy_name", match=MatchValue(value=policy_name)))
    return Filter(must=conditions)

def refresh_indexed_policies_from_qdrant():
    """Rebuild the local (organization, policy_name) -> info cache by scanning Qdrant — picks up
    anything already indexed in a previous session, since the collection persists across runtimes."""
    indexed_policies.clear()
    next_offset = None
    while True:
        points, next_offset = qdrant_client.scroll(
            collection_name=QDRANT_COLLECTION, with_payload=True, with_vectors=False,
            limit=256, offset=next_offset,
        )
        for point in points:
            meta = point.payload.get("metadata", {})
            key = (meta.get("organization"), meta.get("policy_name"))
            info = indexed_policies.setdefault(key, {
                "policy_version": meta.get("policy_version"),
                "source_file": meta.get("source_file"),
                "chunk_count": 0,
            })
            info["chunk_count"] += 1
        if next_offset is None:
            break
    return indexed_policies

refresh_indexed_policies_from_qdrant()
print(f"Found {len(indexed_policies)} already-indexed (organization, policy) pair(s) in Qdrant Cloud.")

def get_indexed_organizations():
    return sorted({organization for organization, _ in indexed_policies})

def get_stored_chunks(organization, policy_name):
    """Return every chunk currently stored in Qdrant Cloud for this (organization, policy_name), in page order."""
    chunks = []
    next_offset = None
    while True:
        points, next_offset = qdrant_client.scroll(
            collection_name=QDRANT_COLLECTION,
            scroll_filter=_org_policy_filter(organization, policy_name),
            with_payload=True, with_vectors=False,
            limit=256, offset=next_offset,
        )
        chunks.extend(points)
        if next_offset is None:
            break
    chunks.sort(key=lambda p: p.payload.get("metadata", {}).get("page", 0))
    return [{"text": p.payload.get("page_content", ""), "metadata": p.payload.get("metadata", {})} for p in chunks]

def index_policy_pdf(pdf_path, organization, policy_name, policy_version):
    """Load, tag, chunk, embed, and store a policy PDF in Qdrant Cloud. Replaces any previous
    version of the SAME policy (same organization + policy_name) — other policies for that org
    are untouched."""
    qdrant_client.delete(
        collection_name=QDRANT_COLLECTION,
        points_selector=FilterSelector(filter=_org_policy_filter(organization, policy_name)),
    )

    pages = load_pdf_as_documents(pdf_path, organization, policy_name, policy_version)
    chunks = chunk_documents(pages)
    policy_vector_store.add_documents(chunks)

    indexed_policies[(organization, policy_name)] = {
        "policy_version": policy_version,
        "source_file": os.path.basename(pdf_path),
        "chunk_count": len(chunks),
    }
    return len(chunks)

print("index_policy_pdf() ready (backed by Qdrant Cloud).")

### 1.4 Organization-filtered retrieval (Qdrant)

In [ ]:
def retrieve_for_organization(question, organization, k=4):
    return policy_vector_store.similarity_search(
        question,
        k=k,
        filter=_org_policy_filter(organization),
    )

print("retrieve_for_organization() ready.")

### 1.5 Strict policy-only prompt and the required no-answer message

Answer only from retrieved context; unsupported questions get exactly: *"I couldn't find this information in the available policies."*

In [ ]:
NO_ANSWER_MESSAGE = "I couldn't find this information in the available policies."

STRICT_POLICY_PROMPT = """You are a policy assistant. Answer the question using ONLY the CONTEXT below,
which comes from official policy documents for {organization}.

Rules:
- Do not use outside/general knowledge to fill in missing details.
- Do not infer a rule the context does not actually state.
- If the question is not about the policy content, or the context does not contain the answer,
  respond with EXACTLY this sentence and nothing else: "{no_answer}"

CONTEXT:
{context}

Question: {question}

Answer:"""

_llm_cache = {}

def get_llm(model_name, temperature=0.0):
    if model_name not in _llm_cache:
        _llm_cache[model_name] = ChatGoogleGenerativeAI(model=model_name, temperature=temperature)
    return _llm_cache[model_name]

def ask_policy_question(question, organization, model_name):
    """Returns the answer text, strictly grounded in retrieved policy context."""
    if organization not in get_indexed_organizations():
        return NO_ANSWER_MESSAGE

    retrieved_docs = retrieve_for_organization(question, organization)
    if not retrieved_docs:
        return NO_ANSWER_MESSAGE

    context = "\n\n".join(doc.page_content for doc in retrieved_docs)
    prompt = STRICT_POLICY_PROMPT.format(
        organization=organization, no_answer=NO_ANSWER_MESSAGE, context=context, question=question,
    )

    llm_for_answer = get_llm(model_name)
    return get_text(llm_for_answer.invoke(prompt)).strip()

print("ask_policy_question() ready.")

### 1.6 Upload & index policies — run this once

This is the **only** place documents get uploaded. Uses `ipywidgets.FileUpload`, which works the same way in Kaggle, Colab, and plain Jupyter — no platform-specific upload API needed. Select every policy PDF you want available (e.g. everything in `Sample Documents`) and click **Index selected PDFs**. Filenames like `01_technova_Leave_Time_Off_Policy_v1_0.pdf` are auto-parsed into organization (`technova`), policy name (`Leave Time Off Policy`), and version (`v1.0`). Right after each PDF is indexed, the actual chunks stored in the vector DB are printed below so you can see what will be retrieved, and the local temp copy is deleted immediately — the PDF only persists as vectors in Qdrant Cloud. Re-run this cell later only if you want to add or replace policies; the Gradio app below has no upload control of its own — it just answers questions against whatever is indexed here.

In [ ]:
import re
import ipywidgets as widgets
from IPython.display import display

# Matches: <seq>_<organization>_<Policy_Name_With_Underscores>_v<major>_<minor>.pdf
FILENAME_PATTERN = re.compile(r"^\d+_([A-Za-z0-9]+)_(.+)_v(\d+)_(\d+)\.pdf$", re.IGNORECASE)

def parse_policy_filename(filename):
    match = FILENAME_PATTERN.match(filename)
    if not match:
        return None
    organization, name_slug, major, minor = match.groups()
    return {
        "organization": organization,
        "policy_name": name_slug.replace("_", " "),
        "policy_version": f"v{major}.{minor}",
    }

def print_stored_chunks(organization, policy_name, preview_len=200):
    """Show what actually landed in the vector DB for this policy, right after indexing it."""
    chunks = get_stored_chunks(organization, policy_name)
    print(f"   Stored chunks in vector DB ({len(chunks)}):")
    for i, doc in enumerate(chunks, start=1):
        text = " ".join(doc.get("text", "").split())
        if len(text) > preview_len:
            text = text[:preview_len].rstrip() + "..."
        page = doc.get("metadata", {}).get("page", "?")
        print(f"     [{i}] page {page}: {text}")

def _iter_uploaded_files(file_upload_value):
    """Normalize ipywidgets.FileUpload.value across versions: a dict {name: {"content": ...}}
    in ipywidgets 7, or a tuple of {"name": ..., "content": ...} dicts in ipywidgets 8+."""
    if isinstance(file_upload_value, dict):
        for name, info in file_upload_value.items():
            yield name, bytes(info["content"])
    else:
        for info in file_upload_value:
            yield info["name"], bytes(info["content"])

def index_uploaded_pdfs(file_upload_value):
    for filename, content in _iter_uploaded_files(file_upload_value):
        with open(filename, "wb") as f:
            f.write(content)
        try:
            meta = parse_policy_filename(filename)
            if meta is None:
                print(f"Skipping '{filename}' — doesn't match the '<seq>_<org>_<Policy_Name>_v<major>_<minor>.pdf' pattern.")
                continue
            chunk_count = index_policy_pdf(filename, meta["organization"], meta["policy_name"], meta["policy_version"])
            print(f"Indexed {meta['organization']} — {meta['policy_name']} {meta['policy_version']}: {chunk_count} chunks")
            print_stored_chunks(meta["organization"], meta["policy_name"])
            print()
        finally:
            # Remove the local temp copy once it's embedded into Qdrant so the PDF doesn't
            # linger on disk — the vector store is the only place the content persists.
            if os.path.exists(filename):
                os.remove(filename)

    print("Indexed organizations:", get_indexed_organizations())

upload_widget = widgets.FileUpload(accept=".pdf", multiple=True, description="Select PDFs")
index_btn_widget = widgets.Button(description="Index selected PDFs", button_style="primary")
output_widget = widgets.Output()

def _on_index_click(_):
    with output_widget:
        output_widget.clear_output()
        if not upload_widget.value:
            print("No files selected — click 'Select PDFs' above first.")
            return
        index_uploaded_pdfs(upload_widget.value)

index_btn_widget.on_click(_on_index_click)
display(upload_widget, index_btn_widget, output_widget)

## 2. Gradio App — Policy Assistant UI

No upload control here — documents are indexed once via Section 1.6. Just pick an **organization** and **LLM model**, then ask a question with the arrow button. Styling is loaded from **`policy_assistant.css`** — make sure that file is uploaded alongside this notebook (same folder), or it falls back to plain default Gradio styling.

In [ ]:
import gradio as gr

def _gradio_version_tuple():
    nums = []
    for part in gr.__version__.split("."):
        digits = "".join(ch for ch in part if ch.isdigit())
        nums.append(int(digits) if digits else 0)
    return tuple(nums[:3])

# Chat data must be role/content dicts from Gradio 4.44+ (whether or not the `type` kwarg itself
# is accepted — 5.x+ made "messages" the only format and REMOVED the `type` param entirely).
GRADIO_SUPPORTS_MESSAGES = _gradio_version_tuple() >= (4, 44, 0)
print(f"Detected Gradio {gr.__version__} — using {'messages' if GRADIO_SUPPORTS_MESSAGES else 'tuples'} chat data format.")

# A small, curated set of chat-capable models for the dropdown (avoids preview/experimental clutter).
# genai.list_models() occasionally times out over Colab's network, so fall back to the curated
# list itself rather than failing the whole UI on a transient network hiccup.
_preferred_models = ["gemini-3.1-flash-lite", "gemini-3.5-flash", "gemini-2.5-flash", "gemini-2.5-pro"]
try:
    _available_model_names = {m.name.split("/")[-1] for m in genai.list_models() if "generateContent" in m.supported_generation_methods}
    available_models = [m for m in _preferred_models if m in _available_model_names] or list(_available_model_names)[:4]
except Exception as e:
    print(f"Could not list models ({e}); using the curated model list as-is.")
    available_models = _preferred_models

def _append_turn(chat_history, user_text, bot_text):
    if GRADIO_SUPPORTS_MESSAGES:
        chat_history.append({"role": "user", "content": user_text})
        chat_history.append({"role": "assistant", "content": bot_text})
    else:
        chat_history.append((user_text, bot_text))
    return chat_history

def do_ask(question, organization, model_name, chat_history):
    chat_history = chat_history or []
    if not question.strip():
        return chat_history, ""
    if not organization:
        return _append_turn(chat_history, question, "Please select an organization first."), ""

    answer = ask_policy_question(question, organization, model_name)
    return _append_turn(chat_history, question, answer), ""

def refresh_organizations():
    return gr.update(choices=get_indexed_organizations())

# Forces light mode regardless of the browser/OS light-dark preference, since Gradio's automatic
# switching otherwise depends on prefers-color-scheme.
FORCE_LIGHT_JS = """
() => {
    document.documentElement.classList.remove('dark');
    document.body.classList.remove('dark');
}
"""

# Styling lives in a separate file — policy_assistant.css — so it can be uploaded/edited
# independently on Kaggle/Colab. It must sit next to this notebook.
CSS_FILE = "policy_assistant.css"
if os.path.exists(CSS_FILE):
    with open(CSS_FILE, "r", encoding="utf-8") as f:
        CUSTOM_CSS = f.read()
    print(f"Loaded styling from '{CSS_FILE}'.")
else:
    CUSTOM_CSS = ""
    print(f"'{CSS_FILE}' not found next to the notebook — using default Gradio styling. "
          f"Upload policy_assistant.css alongside the notebook to get the custom look.")

THEME = gr.themes.Default(
    primary_hue="blue",
    secondary_hue="slate",
    neutral_hue="slate",
    font=[gr.themes.GoogleFont("Inter"), "ui-sans-serif", "system-ui", "sans-serif"],
).set(
    body_background_fill="#f5f6f8",
    body_background_fill_dark="#f5f6f8",
    background_fill_primary="#ffffff",
    background_fill_primary_dark="#ffffff",
    background_fill_secondary="#f8f9fb",
    background_fill_secondary_dark="#f8f9fb",
    border_color_primary="#e3e6ec",
    border_color_primary_dark="#e3e6ec",
    block_background_fill="#ffffff",
    block_background_fill_dark="#ffffff",
    block_border_color="#e3e6ec",
    block_border_color_dark="#e3e6ec",
    block_label_background_fill="#f8f9fb",
    block_label_background_fill_dark="#f8f9fb",
    block_label_text_color="#6b7280",
    block_label_text_color_dark="#6b7280",
    block_title_text_color="#10131a",
    block_title_text_color_dark="#10131a",
    body_text_color="#10131a",
    body_text_color_dark="#10131a",
    body_text_color_subdued="#6b7280",
    body_text_color_subdued_dark="#6b7280",
    input_background_fill="#f8f9fb",
    input_background_fill_dark="#f8f9fb",
    input_border_color="#e3e6ec",
    input_border_color_dark="#e3e6ec",
    button_primary_background_fill="#2563eb",
    button_primary_background_fill_dark="#2563eb",
    button_primary_background_fill_hover="#1d4ed8",
    button_primary_background_fill_hover_dark="#1d4ed8",
    button_primary_text_color="#ffffff",
    button_primary_text_color_dark="#ffffff",
    button_secondary_background_fill="#f8f9fb",
    button_secondary_background_fill_dark="#f8f9fb",
    button_secondary_border_color="#e3e6ec",
    button_secondary_border_color_dark="#e3e6ec",
    button_secondary_text_color="#10131a",
    button_secondary_text_color_dark="#10131a",
    block_radius="14px",
    input_radius="10px",
    block_shadow="0 1px 3px rgba(16,19,26,0.06)",
)

with gr.Blocks(title="Policy Assistant", theme=THEME, css=CUSTOM_CSS, js=FORCE_LIGHT_JS) as demo:
    gr.HTML(
        """
        <div id="app-header">
            <div class="badge">Qdrant Cloud &middot; Gemini</div>
            <h1>Policy Assistant</h1>
            <p>Multi-organization policy Q&amp;A, strictly grounded in documents indexed in Qdrant Cloud.
            Documents are uploaded once via Section 1.6 above — this console only reads what has already been indexed.</p>
        </div>
        """
    )

    with gr.Row(equal_height=False):
        with gr.Column(scale=1, min_width=290, elem_id="sidebar"):
            gr.Markdown('<div class="sidebar-title">Configuration</div>')
            org_dropdown = gr.Dropdown(choices=get_indexed_organizations(), label="Organization")
            llm_dropdown = gr.Dropdown(choices=available_models, value=available_models[0], label="LLM Model")
            refresh_btn = gr.Button("Refresh organizations", size="sm")

        with gr.Column(scale=4, elem_id="main-panel"):
            # The `type` kwarg only exists on Gradio 4.44–4.x (5.x+ removed it, but still expects
            # message dicts by default, so dropping the kwarg there is safe).
            try:
                chatbot = gr.Chatbot(label="Policy Q&A", height=520, type="messages", show_label=False)
            except TypeError:
                chatbot = gr.Chatbot(label="Policy Q&A", height=520, show_label=False)

            with gr.Row():
                question_box = gr.Textbox(
                    placeholder="Ask a question about the selected organization's policy...",
                    scale=8,
                    container=False,
                )
                ask_btn = gr.Button("→", variant="primary", scale=1, min_width=48, elem_classes="send-icon-btn")

    refresh_btn.click(refresh_organizations, None, org_dropdown)
    ask_btn.click(do_ask, [question_box, org_dropdown, llm_dropdown, chatbot], [chatbot, question_box])
    question_box.submit(do_ask, [question_box, org_dropdown, llm_dropdown, chatbot], [chatbot, question_box])

demo.launch(debug=False)

## Recap

```
LLM → Parameters → PDF Loading → Org/Policy Tagging → Chunking → Embeddings
   → Qdrant Cloud (persistent vector store) → Org-Filtered Retrieval → Strict RAG Prompt → Citations → Gradio UI
```

Left for later: conversation history, automatic policy metadata/version detection, DOCX/TXT/Web ingestion, authentication, evaluation dashboard.